In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from pydmd import BOPDMD
import matplotlib.pyplot as plt

pd.set_option('display.max_rows',50)

# -------------------------------------------------------------------------
# 1. DATA ACQUISITION & STATIONARY TRANSFORMATION
# -------------------------------------------------------------------------
# Define a representative basket of S&P 500 Energy Sector stocks
tickers = ['XOM', 'CVX', 'COP', 'EOG', 'DVN', 'OKE', 'MPC', 'SLB']

print(f"Downloading historical data for tickers: {tickers}...")
# Fetch daily closing prices (using a ~40-day lookback window + extra days for lags)
raw_data = yf.download(tickers, start="2026-04-01", end="2026-06-20")['Close']

# Drop any missing values to ensure matrix continuity
raw_data = raw_data.dropna()

# Transform non-stationary prices into stationary daily log returns
df_returns = np.log(raw_data / raw_data.shift(1)).dropna()

[*********************100%***********************]  8 of 8 completed


In [2]:
# -------------------------------------------------------------------------
# 2. TIME-DELAY EMBEDDING (HANKEL MATRIX PREPROCESSING)
# -------------------------------------------------------------------------
# Extract the underlying numpy array: shape is (Time, Assets)
X_raw = df_returns.values

print(f"Number of days in training set = {X_raw.shape[0]}")
print(f"Number of stocks in training set = {X_raw.shape[1]}")

num_assets = X_raw.shape[1]
num_lags = 3  # Number of historical delays/lags to stack
num_snapshots = X_raw.shape[0]

# Manually construct the spatio-temporal Hankel matrix structure
# We stack delayed snapshots vertically to build a rich spatio-temporal row space
hankel_list = []
for i in range(num_lags):
    # Shift time slices to capture memory effects
    slice_data = X_raw[i : num_snapshots - num_lags + 1 + i, :].T
    hankel_list.append(slice_data)

# Combined matrix shape will dynamically grow to: (Assets * Lags, Time - Lags + 1)
X_hankel = np.vstack(hankel_list)

# Split into standard DMD snapshot pairs: X and X_prime (shifted by 1 time step)
X = X_hankel[:, :-1]
X_prime = X_hankel[:, 1:]

print(f"Original matrix shape (Time, Assets): {X_raw.shape}")
print(f"Spatio-Temporal Hankel matrix snapshot shape: {X.shape}")

Number of days in training set = 54
Number of stocks in training set = 8
Original matrix shape (Time, Assets): (54, 8)
Spatio-Temporal Hankel matrix snapshot shape: (24, 51)


In [3]:
t = np.arange(X_hankel.shape[1])
t

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51])

In [4]:
# -------------------------------------------------------------------------
# 3. BOPDMD OPTIMIZATION & FITTING
# -------------------------------------------------------------------------
# Initialize BOPDMD with bagging to mitigate spectral instability and noisy data
bopdmd = BOPDMD(
    svd_rank=6,          # Subspace truncation rank to eliminate asset noise
    num_trials=20,      # Generate an ensemble over 20 randomized data subsets
    trial_size=0.8
)

# Fit the optimized operator to our snapshot series
bopdmd.fit(X_hankel, t)

/home/ongzy/miniconda3/envs/shro2026_environment/lib/python3.12/site-packages/pydmd/bopdmd.py:973: UserWarning: Initial trial of Optimized DMD failed to converge. Consider re-adjusting your variable projection parameters with the varpro_opts_dict and consider setting verbose=True.
  warnings.warn(msg)


In [20]:
t_future = np.linspace(52,60,9)
t_future
forecast_mean, forecast_variance = bopdmd.forecast(t_future)

forecast_mean.real[:8,:].shape


(8, 9)